In [0]:
%run ./utility/logger

In [0]:
dbutils.widgets.text('catalog',"")
dbutils.widgets.text('schema',"")
dbutils.widgets.text('env',"")

In [0]:
catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
env = dbutils.widgets.get('env')

In [0]:
print(schema)

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW sales_incremental AS

SELECT *
FROM commerce_stage_{env}.silver.order_detail_stage

WHERE updated_ts >
(
SELECT COALESCE(MAX(last_processed_ts), TIMESTAMP('1900-01-01'))
FROM commerce_main_{env}.util.etl_control
WHERE table_name = 'fact_sales'
)
""")

In [0]:
spark.sql(f"""
INSERT INTO {catalog}.{schema}.fact_sales
(
order_id,
customer_key,
product_key,
date_key,
quantity,
unit_price,
sales_amount,
created_ts
)

SELECT

o.order_id,

dc.customer_key,

dp.product_key,

dd.date_key,

oi.quantity,

oi.unit_price,

oi.quantity * oi.unit_price AS sales_amount,

current_timestamp()

FROM sales_incremental oi

INNER JOIN commerce_stage_{env}.silver.order_stage o
ON oi.order_id = o.order_id 

INNER JOIN {catalog}.{schema}.dim_customer dc
ON o.customer_id = dc.customer_id
AND dc.is_current = true

INNER JOIN {catalog}.{schema}.dim_product dp
ON oi.product_id = dp.product_id
AND dp.is_current = true

INNER JOIN {catalog}.{schema}.dim_date dd
ON CAST(o.order_date AS DATE) = dd.full_date
""")

In [0]:
spark.sql(f"""
MERGE INTO commerce_main_{env}.util.etl_control tgt

USING
(
SELECT
'fact_sales' AS table_name,
MAX(updated_ts) AS last_processed_ts
FROM commerce_stage_{env}.silver.order_detail_stage
) src

ON tgt.table_name = src.table_name

WHEN MATCHED THEN
UPDATE SET
tgt.last_processed_ts = src.last_processed_ts

WHEN NOT MATCHED THEN
INSERT
(
table_name,
last_processed_ts
)
VALUES
(
src.table_name,
src.last_processed_ts
)
""")